# AI Guardrails Tutorial
## Module 4: Output Guardrails (Validating Model Responses)

In **Module 3** we locked the front door: prompt-injection payloads were blocked before reaching the model, and PII was redacted on the way in.

This module guards the **exit** - everything the model generates *before it reaches the user*. You will build three output rails:

1. **Schema compliance (4.1)** - force answers into a rigid Pydantic structure, with automatic **re-asking** when the model drifts.
2. **Toxicity screening (4.2)** - catch, strip, or block abusive generations with `guardrails/toxic_language`.
3. **Grounding / provenance (4.3)** - verify that answers are actually supported by the source text you provided, instead of hallucinated.

> **The core principle of this module:** output guardrails are your last line of defense. They run *after* the model, so they cannot stop a bad generation from happening - they decide what is allowed to leave your system.

Sections 4.1-4.3 are fully deterministic and run offline with no API key. Each section ends with an optional live-model demo.

## Setup & Configuration

We reuse the same `.env` conventions as Modules 1-3, so everything runs against **OpenAI or a local Ollama server** without changes.

In [ ]:
# =============================================================================
# Initialize: load configuration from the .env file
# OpenAI and Ollama are both supported - just set USE_OLLAMA in .env
# =============================================================================
import json
import os

from dotenv import load_dotenv

load_dotenv(".env")

MODEL_NAME = os.getenv("LLM_MODEL")
BASE_URL = os.getenv("BASE_URL")
API_KEY = os.getenv("API_KEY")
USE_OLLAMA = os.getenv("USE_OLLAMA", "false").lower() == "true"
DEBUG_MODE = os.getenv("DEBUG_MODE", "false").lower() == "true"

if USE_OLLAMA:
    MODEL_NAME = MODEL_NAME or "llama3"
    # The /v1 suffix matters: we talk to the OpenAI-compatible API, not the
    # native Ollama API. BASE_URL in .env should include it.
    base_url = BASE_URL or "http://localhost:11434/v1"
    print("[INFO] Provider : Ollama")
else:
    MODEL_NAME = MODEL_NAME or "gpt-4o-mini"
    base_url = BASE_URL  # None -> use the OpenAI default endpoint
    if API_KEY:
        os.environ["OPENAI_API_KEY"] = API_KEY
    print("[INFO] Provider : OpenAI-compatible endpoint")

print(f"[INFO] Model     : {MODEL_NAME}")
print(f"[INFO] Base URL  : {base_url or '(provider default)'}")
print(f"[INFO] API key   : {'configured' if API_KEY else 'NOT SET'}")

# Everything in sections 4.1-4.3 is deterministic and needs NO API key.
# Only the optional live cells talk to a real model.
LLM_READY = bool(API_KEY) or USE_OLLAMA

In [ ]:
# =============================================================================
# Keep cell output clean: the guardrails library phones home by default
# 1) Opt out of anonymous usage metrics and OpenTelemetry tracing.
# 2) Mute the metrics exporter logger - a fresh Guard() re-reads the user
#    config on construction, so a stray span can otherwise still be queued
#    and retried against an unreachable endpoint when you are offline.
# 3) Silence model-download noise from Hugging Face.
# =============================================================================
import logging
import os
import warnings

from guardrails import settings

settings.rc.enable_metrics = False  # no anonymous usage metrics
settings.disable_tracing = True     # no OpenTelemetry spans

os.environ.setdefault("HF_HUB_VERBOSITY", "error")  # Hub client at import time
logging.getLogger(
    "opentelemetry.exporter.otlp.proto.http.trace_exporter"
).setLevel(logging.CRITICAL)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*unauthenticated requests.*")

print("Telemetry disabled; exporter and download noise muted.")

In [ ]:
# =============================================================================
# OpenAI-compatible client for the optional live cells (same conventions as
# Modules 1-3). A placeholder key keeps construction valid; the client is only
# *used* when LLM_READY is True.
# =============================================================================
from openai import OpenAI

client = OpenAI(base_url=base_url, api_key=API_KEY or "not-set")


def chat(messages, temperature=0.0):
    '''Live-model helper: returns the reply text, or None when unreachable.'''
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME, messages=messages, temperature=temperature
        )
        return response.choices[0].message.content
    except Exception as exc:
        print("Live model unreachable:", type(exc).__name__, "-", str(exc)[:120])
        print("Start your server (or fix .env), then re-run this cell.")
        return None


print("Live client ready for:", base_url or "https://api.openai.com/v1")

## 4.1 Structural Integrity & JSON Validation

Your application does not consume prose - it consumes fields. When an LLM is asked to "return JSON", three things go wrong in practice:

| Failure mode | What it looks like |
|--------------|--------------------|
| Prose wrapping | `Sure! Here is your JSON: {...} Hope it helps!` |
| Wrong shape | Missing keys, wrong types (`"days": "two"`), unknown keys |
| Mirage fidelity | Perfectly valid JSON whose *values* are invented (4.3's problem) |

Pydantic gives you the *contract*; **Guardrails gives you the enforcement**. `Guard.for_pydantic()` turns a model class into a JSON Schema guard that:

1. Parses the model's answer as JSON (tolerating prose wrapping).
2. Validates it against the schema, producing exact field-level errors.
3. On failure, calls the LLM again with the errors attached - the **re-ask** loop - and re-validates the answer.

The demo below is deterministic: a scripted stand-in for the LLM lets us watch the re-ask machinery without burning tokens.

In [ ]:
# =============================================================================
# 4.1 Step 1: define the contract - a rigid Pydantic model
# Field descriptions become part of the JSON Schema sent back to the model
# during a re-ask, so write them for the model's benefit.
# =============================================================================
from typing import List

from pydantic import BaseModel, Field


class DayPlan(BaseModel):
    day: int = Field(description="Day number, starting at 1")
    title: str = Field(description="Short human-readable theme for the day")
    activities: List[str] = Field(
        min_length=1, max_length=5, description="1-5 activities for this day"
    )


class Itinerary(BaseModel):
    destination: str = Field(description="City the trip is planned for")
    days: int = Field(description="Total number of days")
    plan: List[DayPlan] = Field(description="One entry per day")
    estimated_budget_usd: float = Field(description="Rough budget in USD")


print("Itinerary schema ready. Required fields:",
      ", ".join(Itinerary.model_json_schema()["required"]))

In [ ]:
# =============================================================================
# 4.1 Step 2: turn the model class into a guard and try both paths
# =============================================================================
from guardrails import Guard

itinerary_guard = Guard.for_pydantic(Itinerary)
# Opt out of anonymous usage metrics for this guard (enabled by default).
itinerary_guard.configure(allow_metrics_collection=False)

# --- The happy path: a valid payload passes through -------------------------
good_payload = {
    "destination": "Paris",
    "days": 2,
    "plan": [
        {"day": 1, "title": "Museums", "activities": ["Louvre"]},
        {"day": 2, "title": "Food", "activities": ["Market tour"]},
    ],
    "estimated_budget_usd": 600,
}
outcome = itinerary_guard.validate(json.dumps(good_payload))
print("valid payload   :", outcome.validation_passed)
# validated_output is a plain dict - rehydrate it into your Pydantic model:
parsed = Itinerary.model_validate(outcome.validated_output)
print("rehydrated      :", type(parsed).__name__, "| destination =", parsed.destination)

# --- The real world: one payload, three separate problems -------------------
bad_payload = {
    "destination": "Paris",
    "days": "two",                              # string instead of integer
    "plan": [{"day": 1, "title": "Museums"}],   # 'activities' is missing
}                                               # 'estimated_budget_usd' is missing
outcome = itinerary_guard.validate(json.dumps(bad_payload))
print()
print("invalid payload :", outcome.validation_passed)
print("field-level errors from the reask object:")
print(outcome.reask.fail_results[0].error_message)

In [ ]:
# =============================================================================
# 4.1 Step 3: automatic re-asking (deterministic demo)
# llm_api contract: called with OpenAI-style messages (a list of dicts with
# 'role' and 'content'); must return a string. Guardrails sends the failed
# output plus the exact validation errors on each re-ask.
# =============================================================================
reask_calls = []


def scripted_llm(*, messages, **kwargs):
    '''Stand-in for an LLM: every re-ask returns a corrected itinerary.'''
    reask_calls.append(messages)
    return (
        '{"destination": "Paris", "days": 2, "plan": ['
        '{"day": 1, "title": "Museums", "activities": ["Louvre"]}, '
        '{"day": 2, "title": "Food", "activities": ["Market tour"]}], '
        '"estimated_budget_usd": 600}'
    )


messy_first_answer = (
    "Sure! Here is your itinerary: "
    '{"destination": "Paris", "days": 2, "plan": [{"day": 1, "title": "Museums"}]} '
    "Hope you enjoy!"
)

outcome = itinerary_guard.parse(
    llm_output=messy_first_answer,  # candidate #1: validate this first
    llm_api=scripted_llm,           # used for re-asks when #1 fails
    num_reasks=1,                   # how many correction attempts are allowed
)

print("passed after re-ask :", outcome.validation_passed)
print("validated_output    :", json.dumps(outcome.validated_output))
print("llm_api calls       :", len(reask_calls))
print()

# What did the model actually receive on the re-ask?
first_reask = reask_calls[0]
print("re-ask message roles:", [m["role"] for m in first_reask])
user_msg = next(m for m in first_reask if m["role"] == "user")["content"]
print("re-ask user message (error section):")
print(user_msg.split("Given below is a JSON Schema")[0].strip())

In [ ]:
# =============================================================================
# 4.1 Step 4: when re-asking runs out
# A model that never produces valid JSON must not crash your application.
# =============================================================================
def stubborn_llm(*, messages, **kwargs):
    return "I am sorry, I cannot produce structured JSON today."


final = itinerary_guard.parse(
    llm_output=messy_first_answer,
    llm_api=stubborn_llm,
    num_reasks=2,
)

print("passed           :", final.validation_passed)
print("validated_output :", final.validated_output)
print("last failure     :", final.reask.fail_results[-1].error_message if final.reask else "-")
print("parser error     :", final.error)
print()
print("passed=False + validated_output=None is your cue to fall back: retry later,")
print("serve a canned response, or escalate - never forward raw broken output.")

### Where should structure be enforced?

Structured output is a first-class feature of modern providers, and it pays to know which layer guarantees what:

| Approach | Guarantees | The loop lives in | Notes |
|----------|-----------|-------------------|-------|
| Provider JSON mode (`response_format="json_object"`) | Valid JSON *syntax* only | nowhere - no retry | Cheapest; adds no schema compliance |
| Provider tool/function calling | Syntax + the tool's schema | provider / SDK | Provider-specific; no custom business rules |
| **Guardrails `parse()` + validators** | Syntax + schema + **any validator chain** | your app, visible via `llm_api` messages and `outcome.reask` | Model-agnostic; works with local models; the re-ask audit trail is yours |

A robust production setup often layers them: provider tool calling as the fast path, Guardrails re-asking as the safety net for whatever slips through.

In [ ]:
# =============================================================================
# 4.1 Lab (optional): the same re-ask loop with a live model
# Gated behind LLM_READY. Small local models fail impressively here - which is
# exactly what the exhaustion path above prepared you for. Run it more than
# once: whether the model self-corrects on the re-ask is luck, and the guard
# keeps the *system* correct either way.
# =============================================================================
if not LLM_READY:
    print("Skipping live call - set API_KEY or USE_OLLAMA=true in .env to enable it.")
else:
    raw = chat([
        {
            "role": "system",
            "content": "You are a travel planning assistant. Respond with ONLY a JSON object.",
        },
        {
            "role": "user",
            "content": "Plan a 2-day trip to Paris for a family of four.",
        },
    ])
    if raw is not None:
        print("raw model output (first 300 chars):")
        print(raw[:300] + ("..." if len(raw) > 300 else ""))

        def live_llm(*, messages, **kwargs):
            '''llm_api adapter: translate Guardrails messages into a chat call.'''
            reply = chat([{"role": m["role"], "content": m["content"]} for m in messages])
            if reply is None:
                raise RuntimeError("live model unreachable")
            return reply

        # num_reasks=1 keeps the re-ask prompt small enough for tiny local
        # models: a 2048-token context fills up fast once schema and error
        # text are added.
        try:
            live_outcome = itinerary_guard.parse(
                llm_output=raw, llm_api=live_llm, num_reasks=1
            )
        except Exception as exc:  # e.g. a local model with a tiny context window
            print()
            print("live re-ask could not complete:", type(exc).__name__)
            print(str(exc)[:280])
        else:
            print()
            print("passed       :", live_outcome.validation_passed)
            print("validated    :", json.dumps(live_outcome.validated_output, indent=2)
                  if live_outcome.validation_passed else None)
            if live_outcome.reask is not None:
                print("last failure :", live_outcome.reask.fail_results[0].error_message)

## 4.2 Content Moderation & Toxic Output Prevention

Input screens handle hostile *users*; they do nothing about hostile *generations*. A model can insult, threaten, or flame in response to entirely benign prompts - and every sentence it produces is something your brand ships.

`guardrails/toxic_language` runs the Detoxify classifier **locally** (no API calls, nothing leaves the machine) and scores each sentence. When a sentence crosses the toxicity threshold, the validator:

- reports exactly which sentences were toxic, and
- computes a `fix_value`: the text with the toxic sentences removed.

**Policy is your choice, not the validator's** - the `on_fail` action decides what happens to the output. We will exercise all of it.

In [ ]:
# =============================================================================
# 4.2 Step 1: load the toxicity validator three ways
# First instantiation downloads a small model (~45 MB) and loads torch, so it
# can take up to a minute - later instantiations are fast.
# threshold=0.5 is the default: flag a sentence when its toxicity score > 0.5.
# =============================================================================
from guardrails_ai.toxic_language import ToxicLanguage

toxic_reporter = ToxicLanguage(threshold=0.5, validation_method="sentence", on_fail="noop")
toxic_fixer = ToxicLanguage(threshold=0.5, validation_method="sentence", on_fail="fix")
toxic_blocker = ToxicLanguage(threshold=0.5, validation_method="sentence", on_fail="exception")

print("ToxicLanguage ready:", toxic_reporter.rail_alias)
print("Full mode (whole-text scoring) is also available: validation_method='full'.")

In [ ]:
# =============================================================================
# 4.2 Step 2: battery - what must be flagged, what must stay clean
# The benign rows include emotionally loaded but non-abusive language; a good
# toxicity filter must not get fooled by intensity alone.
# =============================================================================
from guardrails.validator_base import FailResult

samples = [
    ("allow", "Thanks for reaching out - I will put together a great itinerary for you."),
    ("block", "You are an idiot and this itinerary is garbage."),
    ("block", "Shut up, you stupid loser. Nobody wants your opinion."),
    ("block", "I will destroy you if you don't book this hotel."),
    ("allow", "The weather was awful, but the views were breathtaking."),
    ("allow", "This hidden-gem itinerary is an absolute killer for food lovers."),
]

flagged = allowed = 0
print(f"{'EXPECT':6} {'VERDICT':8} DETAIL / SAMPLE")
for expected, text in samples:
    # ToxicLanguage requires a metadata dict argument; an empty one is fine.
    result = toxic_reporter.validate(text, {})
    if isinstance(result, FailResult):
        flagged += 1
        fix = (result.fix_value or "(everything removed)").replace("\n", " | ")
        print(f"{expected:6} {'TOXIC':8} fix keeps: {fix[:56]!r}")
    else:
        allowed += 1
        print(f"{expected:6} {'clean':8} {text[:60]!r}")

print()
print(f"summary: {flagged}/3 toxic samples flagged, {allowed}/3 benign samples allowed.")

In [ ]:
# =============================================================================
# 4.2 Step 3: the fix policy - repair the text and let it flow on
# =============================================================================
from guardrails import Guard

fixing_guard = Guard().use(toxic_fixer)
fixing_guard.configure(allow_metrics_collection=False)

outcome = fixing_guard.validate("Shut up, you stupid loser. Nobody wants your opinion.")
print("validation_passed :", outcome.validation_passed)
print("validated_output  :", repr(outcome.validated_output))
print()
print("The toxic sentence was deleted and the useful sentence survived - the fix")
print("action repairs the output instead of discarding the whole generation.")

In [ ]:
# =============================================================================
# 4.2 Step 4: the exception policy - zero tolerance, fail loudly
# =============================================================================
from guardrails.errors import ValidationError

strict_guard = Guard().use(toxic_blocker)
strict_guard.configure(allow_metrics_collection=False)

clean = strict_guard.validate("Happy to help plan your trip!")
print("clean text  -> passed:", clean.validation_passed)

try:
    strict_guard.validate("You are an idiot.")
    print("no exception raised (unexpected!)")
except ValidationError as exc:
    print("toxic text  -> blocked with ValidationError:")
    print(exc)

In [ ]:
# =============================================================================
# 4.2 Step 5: sentence mode vs full mode
# Sentence mode pinpoints offenders and builds a surgical fix_value; full mode
# judges the text as a whole (no per-sentence repair available).
# =============================================================================
mixed = "Shut up, you stupid loser. Nobody wants your opinion."

by_sentence = ToxicLanguage(
    threshold=0.5, validation_method="sentence", on_fail="noop"
).validate(mixed, {})
by_full = ToxicLanguage(
    threshold=0.5, validation_method="full", on_fail="noop"
).validate(mixed, {})

print("sentence mode ->", "TOXIC" if isinstance(by_sentence, FailResult) else "clean")
if isinstance(by_sentence, FailResult):
    print("  fix_value keeps only the clean sentences:", repr(by_sentence.fix_value))

print("full mode     ->", "TOXIC" if isinstance(by_full, FailResult) else "clean")
if isinstance(by_full, FailResult):
    print("  error_message:", " ".join(by_full.error_message.split()))

### Choosing an `on_fail` policy

| `on_fail` | Behavior | Use when |
|-----------|----------|----------|
| `noop` | Report the failure; the output passes through unchanged | You want telemetry / alerting without changing behavior |
| `fix` | Replace flagged sentences with `fix_value` (toxic parts removed) | Chat UIs and drafts where salvageable content should survive |
| `exception` | Raise `ValidationError`; nothing reaches the user | Zero-tolerance surfaces: kids' products, regulated messaging |
| `reask` | Send the failure back to the model for another attempt | Interactive agents with a capable model (see 4.1's loop) |

A common production layout: `fix` for user-visible text, `noop` plus alerting for logging pipelines - and `exception` wherever your policy is genuinely zero-tolerance.

In [ ]:
# =============================================================================
# 4.2 Lab (optional): a live model asked to be rude
# Run it a few times - different generations get different verdicts and fixes.
# The point is that the rail, not luck, decides what the user sees.
# =============================================================================
if not LLM_READY:
    print("Skipping live call - set API_KEY or USE_OLLAMA=true in .env to enable it.")
else:
    generation = chat(
        [
            {
                "role": "user",
                "content": (
                    "You are a grumpy hotel concierge. Reply to this guest in the most "
                    "insulting way you can: 'Is this really the best hotel in Paris?'"
                ),
            }
        ],
        temperature=0.7,
    )
    if generation is not None:
        generation = generation.strip()
        print("model said :", generation)
        verdict = toxic_reporter.validate(generation, {})
        if isinstance(verdict, FailResult):
            safe_text = toxic_fixer(generation)
            print("verdict    : TOXIC -> the user would see:", repr(safe_text))
        else:
            print("verdict    : clean (this time)")

## 4.3 Grounding & Fact-Checking Controls

A hallucination is not a lie - it is a fluent, confident sentence that the sources never said. A schema check will not catch it (the JSON is valid) and a toxicity check will not either (the tone is fine). What you *can* do is measure whether each sentence is **close to something in the source text** you gave the model.

`guardrails/provenance_embeddings` does exactly that, entirely locally:

1. Splits the model's answer into sentences.
2. Embeds each sentence (via an `embed_function` you provide) and compares it to embedded **chunks of your source documents**.
3. Flags sentences whose cosine **distance** to their nearest source chunk exceeds a threshold.

> **Note on validator choice:** the outline mentioned `guardrails/provenance_llm`. That package pins an older `litellm`, which conflicts with this shared environment, so we use `provenance_embeddings` - the outline's "similar semantic similarity metric". Same idea, fully local, one less dependency headache. This mirrors Module 3's lesson: check a dependency before you trust it.

**Threshold semantics - memorize this one:** the threshold is a *distance*, so **lower is stricter**. `0.2` flags nearly everything; `0.8` waves hallucinations through. The right value is data-dependent, which is why we calibrate it below.

In [ ]:
# =============================================================================
# 4.3 Step 1: the "knowledge base" and the embedding function
# ProvenanceEmbeddings needs an embed_function: any callable that maps a list
# of strings to vectors. We use a small local sentence-transformers model
# (~90 MB download on first run); in production you can swap in a hosted
# embeddings API here without changing anything else.
# =============================================================================
from guardrails_ai.provenance_embeddings import ProvenanceEmbeddings
from sentence_transformers import SentenceTransformer

SOURCES = [
    "The Aurora Skyline Hotel has 120 rooms and opened in 1998.",
    "The hotel's rooftop bar serves breakfast from 7:00 to 10:30.",
    "The Aurora Skyline Hotel is a 10-minute walk from Central Station.",
]

_embedder = SentenceTransformer("paraphrase-MiniLM-L6-v2")


def embed_function(texts):
    '''Embed a list of strings (the contract ProvenanceEmbeddings expects).'''
    return _embedder.encode(texts)


provenance_validator = ProvenanceEmbeddings(threshold=0.4, on_fail="noop")
print("Embedder loaded; validator ready:", provenance_validator.rail_alias)

In [ ]:
# =============================================================================
# 4.3 Step 2: see the raw numbers
# Distance from each claim to its best-matching source chunk (0 = identical).
# Supported means distance < threshold; anything at/above 0.4 gets flagged.
# =============================================================================
battery = [
    ("grounded", "The Aurora Skyline Hotel opened in 1998."),
    ("grounded", "Breakfast is served on the rooftop every morning from 7:00 to 10:30."),
    ("invented", "The hotel has a rooftop swimming pool and a spa."),
    ("invented", "It also has a casino."),
    ("true-but-distant", "Guests can reach the hotel in ten minutes on foot from Central Station."),
]

print(f"{'EXPECTED':17} {'DIST':>6}  NEAREST SOURCE CHUNK")
for expected, sentence in battery:
    match = provenance_validator.query_vector_collection(
        text=sentence, k=1, sources=SOURCES, embed_function=embed_function
    )
    chunk, distance = match[0]
    print(f"{expected:17} {distance:6.3f}  {chunk[:54]!r}")

print()
print("threshold = 0.4: sentences with distance >= 0.4 are flagged as unsupported.")

In [ ]:
# =============================================================================
# 4.3 Step 3: run the validator - verdicts, precise failures, and repairs
# =============================================================================
claims = [
    "The Aurora Skyline Hotel opened in 1998.",
    "Breakfast is served on the rooftop every morning from 7:00 to 10:30.",
    "The hotel has a rooftop swimming pool and a spa.",
    "The Aurora Skyline Hotel opened in 1998. It also has a casino.",
    "Guests can reach the hotel in ten minutes on foot from Central Station.",
]

for claim in claims:
    # The validator writes into the metadata dict - use a fresh one per call.
    metadata = {"sources": SOURCES, "embed_function": embed_function}
    result = provenance_validator.validate(claim, metadata)
    if isinstance(result, FailResult):
        print("UNSUPPORTED:", claim)
        print("  " + result.error_message.replace("\n", "\n  "))
        kept = (result.fix_value or "(nothing)").replace("\n", " | ")
        print("  fix keeps :", kept)
    else:
        print("supported  :", claim)
    print()

In [ ]:
# =============================================================================
# 4.3 Step 4: which sentence was grounded in which source chunk?
# The validator annotates the metadata dict it was given, sentence by
# sentence - perfect material for citations in a UI.
# =============================================================================
metadata = {"sources": SOURCES, "embed_function": embed_function}
mixed_claim = "The Aurora Skyline Hotel opened in 1998. It also has a casino."
result = provenance_validator.validate(mixed_claim, metadata)

print("verdict:", "UNSUPPORTED" if isinstance(result, FailResult) else "supported")
print()
print("supported sentences (sentence -> matched source chunk):")
for sentence, chunk in metadata["supported_sentences"]:
    print(f"  - {sentence}")
    print(f"      <- {chunk}")
print("unsupported sentences:")
print("  " + metadata["unsupported_sentences"].replace("\n", "\n  "))

In [ ]:
# =============================================================================
# 4.3 Step 5: calibrate the threshold (the one knob you must tune)
# Lower = stricter. Rerun this sweep on YOUR corpus before trusting any value.
# =============================================================================
sweep_claims = [
    ("grounded", "The Aurora Skyline Hotel opened in 1998."),
    ("grounded", "Breakfast is served on the rooftop every morning from 7:00 to 10:30."),
    ("invented", "The hotel has a rooftop swimming pool and a spa."),
    ("invented", "It also has a casino."),
    ("true-but-distant", "Guests can reach the hotel in ten minutes on foot from Central Station."),
]

thresholds = (0.2, 0.3, 0.4, 0.5)
print(f"{'CLAIM TYPE':17} " + "  ".join(f"t={t}" for t in thresholds))
for expected, sentence in sweep_claims:
    marks = []
    for t in thresholds:
        probe = ProvenanceEmbeddings(threshold=t, on_fail="noop")
        result = probe.validate(
            sentence, {"sources": SOURCES, "embed_function": embed_function}
        )
        marks.append("FLAG" if isinstance(result, FailResult) else "pass")
    print(f"{expected:17} " + "  ".join(f"{m:<5}" for m in marks))

### Reading the calibration table

Two truths hide in those numbers:

1. **Distance measures wording, not truth.** The grounded paraphrase scored ~0.19 - closer to its source than some verbatim sentences - while a *true but differently-worded* claim ("ten minutes on foot" vs "10-minute walk") scored ~0.43 and gets flagged. Provenance is a circuit breaker for *unsupported wording*, not an oracle of truth.
2. **The threshold is a precision/recall dial.** Too low and honest paraphrases are rejected; too high and fabrications slip through. For this fact sheet, 0.4 separates the two classes - but that number is calibrated for *this* corpus, this embedder, and this phrasing mix.

For production: pair the check with a re-ask (4.1's loop) so the model can repair an ungrounded sentence, and keep a last-resort fallback ("I could not verify that from the provided sources").

In [ ]:
# =============================================================================
# 4.3 Step 6: wire provenance into a Guard - two policy flavors
# =============================================================================
claim = "The Aurora Skyline Hotel opened in 1998. It also has a spa."

noop_guard = Guard().use(ProvenanceEmbeddings(threshold=0.4, on_fail="noop"))
noop_guard.configure(allow_metrics_collection=False)
outcome = noop_guard.validate(
    claim, metadata={"sources": SOURCES, "embed_function": embed_function}
)
print("noop -> passed:", outcome.validation_passed)
print("        output:", repr(outcome.validated_output))
print()

fix_guard = Guard().use(ProvenanceEmbeddings(threshold=0.4, on_fail="fix"))
fix_guard.configure(allow_metrics_collection=False)
outcome = fix_guard.validate(
    claim, metadata={"sources": SOURCES, "embed_function": embed_function}
)
print("fix  -> output:", repr(outcome.validated_output))
print("        (only sentences with a matching source chunk survive)")

In [ ]:
# =============================================================================
# 4.3 Lab (optional): catch a live hallucination
# The model answers from the fact sheet; the rail decides whether the answer
# is grounded. Note: provenance checks whether the SOURCES support the
# wording - not whether the answer is true. An unsupported "no" is still
# unsupported.
# =============================================================================
if not LLM_READY:
    print("Skipping live call - set API_KEY or USE_OLLAMA=true in .env to enable it.")
else:
    question = "Does the hotel have a rooftop swimming pool and a casino?"
    answer = chat(
        [
            {
                "role": "system",
                "content": "You are a hotel concierge. Answer in one or two short sentences.",
            },
            {
                "role": "user",
                "content": "Context:\n" + "\n".join(SOURCES) + "\n\nQuestion: " + question,
            },
        ],
        temperature=0.2,
    )
    if answer is not None:
        answer = answer.strip()
        print("model answer:", answer)
        metadata = {"sources": SOURCES, "embed_function": embed_function}
        result = provenance_validator.validate(answer, metadata)
        if isinstance(result, FailResult):
            print("verdict: UNSUPPORTED - some sentences are not grounded in the fact sheet")
            print("  " + result.error_message.replace("\n", "\n  "))
        else:
            print("verdict: supported by the fact sheet")

## Summary: What We Achieved in Module 4

### Key Concepts Learned

1. **Contracts beat prose** - `Guard.for_pydantic()` turns a Pydantic model into an enforced JSON Schema, with field-level errors instead of vague complaints.

2. **Re-asking is a bounded loop** - failures are sent back to the model with the exact errors attached; `num_reasks` caps the attempts, and an exhausted loop is a *designed* outcome, not a crash.

3. **Toxicity screening is local and sentence-aware** - `guardrails/toxic_language` scores each sentence and can surgically remove offenders via `fix_value`.

4. **Policy belongs to the application** - the same toxic flag becomes a redaction, a hard block, or a re-ask depending on `on_fail` and your product's risk appetite.

5. **Grounding is measurable, not mystical** - sentence-level embedding distance against source chunks catches invented claims, and the threshold is a precision/recall dial you calibrate per corpus.

6. **Similarity is not truth** - provenance flags *unsupported wording*; true paraphrases can be flagged and clever restatements of falsehoods can pass. Output rails reduce risk; they do not remove judgment.

### The Output Pipeline We Built

| Stage | Mechanism | On failure |
|-------|-----------|-----------|
| Structural integrity | `Guard.for_pydantic(Itinerary)` + re-ask loop | Re-ask up to `num_reasks`, then fall back gracefully |
| Toxicity | `guardrails/toxic_language` (local Detoxify) | `fix` removes toxic sentences; `exception` blocks |
| Grounding | `guardrails/provenance_embeddings` (local MiniLM embedder) | Flag unsupported sentences; `fix` keeps only grounded ones |

| Verification | Result |
|--------------|--------|
| Valid itinerary payload | passes; `validated_output` rehydrates into the Pydantic model |
| Broken payload | rejected with per-field errors (`days`, `plan[0].activities`, `estimated_budget_usd`) |
| Scripted re-ask | one correction attempt turns the broken answer into a passing one |
| Stubborn model | `passed=False`, `validated_output=None` - a graceful, handleable outcome |
| Toxicity battery | 3/3 abusive samples flagged; 3/3 intense-but-benign samples allowed |
| Mixed toxic sentence | `fix` deletes only the toxic sentence; the useful one survives |
| Provenance battery | grounded claims pass; invented claims flagged; mixed claims repaired sentence-by-sentence |
| Threshold sweep | 0.4 separates grounded from invented on this fact sheet - and flags one honest paraphrase, honestly shown |

### Next Steps in Module 5

We now own both ends of the pipeline. Module 5 assembles everything into one production-shaped `pydantic_ai.Agent`: input screening, structured output with validation, toxicity and grounding checks wired into the agent's own retry machinery - and a resiliency test loop that feeds safe, toxic, and malicious prompts to prove the whole assembly degrades gracefully.

### Assignment

1. Add `Field(gt=0)` to `estimated_budget_usd` and push a negative budget through `itinerary_guard.validate()`. Read the re-ask message the model would have received - could a model reasonably fix this?
2. Raise the toxicity threshold to `0.7` and rerun the battery. Which blocked sample slips through? That is the precision/recall trade-off in miniature.
3. Rerun the 4.3 sweep with a different embedder (for example `all-MiniLM-L6-v2`) and compare distances. Which threshold would you pick, and what does that tell you about publishing "a" threshold as universal?